# Piloto de Alda-1: um campo de R1

**Papel nesta entrega:** Suplementar: piloto que antecedeu a ampliacao

**Origem:** https://colab.research.google.com/drive/1chvkwZbqYezhS6_u-FA_0slWq2MkfGgk

Copia de publicacao organizada em 7 out. 2026. Saidas, contadores e metadados de sessao foram removidos; isso nao significa que a campanha original nao tenha sido executada. Nenhum resultado novo foi produzido na curadoria. O registro de proveniencia relaciona codigo, fonte e alteracoes de configuracao.

Preserva a selecao documental do piloto. As impressoes preliminares nao substituem a ficha humana final de R1-R3. A pasta original do piloto permanece necessaria para o caderno seguinte.

# Inspeção biológica dos perfis e das imagens | quatro locais

**IMTM, FMP, MEDINA e USC, HepG2. Etapa descritiva posterior aos modelos.**

A magnitude de um perfil não informa, sozinha, que estruturas mudaram ou se todas as medidas mudaram na mesma direção. Este caderno liga as coordenadas dos poços e os valores com sinal as imagens originais, incluindo controles e casos discordantes. Não seleciona novos candidatos, não altera a regra, não treina modelos e não le R4, nem mesmo dos locais em que a reserva ja foi consultada.

**Estado da entrega:** as tabelas numéricas e a seleção de poços foram executadas com fontes reais e verificadas. A conexão com o S3 não estava disponivel no ambiente de preparação: nenhuma imagem real foi baixada ou inspecionada nesta entrega. O download remoto e a compatibilidade dos arquivos reais de imagem precisam ser confirmados neste Colab. Os testes locais do leitor utilizaram exemplos artificiais separados; eles não são resultados biológicos.

O lote inicial é pequeno: Alda-1 em R1 nos quatro locais e dois DMSO por placa/réplica. A escolha serve para testar toda a cadeia de identidade e visualização; não é uma estimativa da reprodutibilidade entre réplicas. O roteiro completo inclui os seis casos previamente discutidos e pode incluir os quatro casos de auditoria, sem usar a aparencia das imagens como filtro.


## 1. Ambiente: somente leitura de tabelas e imagens

Este caderno não precisa instalar XGBoost ou SHAP. Bibliotecas ja disponiveis não são atualizadas deliberadamente. Se for necessario instalar uma dependencia, reinicie a sessão antes de continuar, para evitar a mistura de bibliotecas antigas na memória com arquivos novos. Depois repita esta célula.


In [ ]:
import sys, subprocess, importlib.util, importlib.metadata
needed={'numpy':'numpy','pandas':'pandas','PIL':'Pillow','requests':'requests','tifffile':'tifffile','imagecodecs':'imagecodecs'}
missing=[pkg for mod,pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','--upgrade-strategy','only-if-needed',*missing])
    raise RuntimeError('DEPENDENCIAS INSTALADAS. Reinicie a sessao e repita esta celula antes de continuar.')
import numpy, pandas, PIL, requests, tifffile, imagecodecs
VERSOES={pkg:importlib.metadata.version(pkg) for pkg in needed.values()}
print('AMBIENTE PRONTO', VERSOES)


## 2. Ficha da inspeção

`MODO='piloto'` escolhe Alda-1 em R1 e o primeiro campo comum documentado. `MODO='completo'` escolhe os seis casos em R1-R3 e três campos igualmente espaçados na lista de campos comuns. Não são escolhidas células ou campos mais bonitos. Os controles são dois DMSO sorteados por uma ordem hash fixa da coordenada, independentemente de qualquer medida.

O limite inicial é 1.024 MiB. Se o lote completo exceder esse volume, o caderno para antes do download. O uso de campos reduzidos não substitui a revisão de todos os campos para uma conclusao definitiva.

O arquivo de fontes deve permanecer na pasta junto a este notebook. Cada execução grava uma pasta nova. Para retomar, informe a pasta exata; parâmetros e fontes devem continuar iguais.


In [ ]:
from pathlib import Path
from google.colab import drive
from IPython.display import display, Markdown, Image as IPyImage
drive.mount('/content/drive')
RAIZ=Path('/content/drive/MyDrive/mba_tcc/comparacao_4_locais_HepG2/inspecao_biologica_4_locais')
FONTES_ZIP=RAIZ/'fontes_inspecao_biologica.zip'
MODO='piloto'  # 'piloto' ou 'completo'
INCLUIR_EXTREMOS_QC=False  # somente no modo completo; nao cria candidatos
LIMITE_MB=1024
BAIXAR_IMAGENS=True
RETOMAR_PASTA=''  # vazio inicia nova execucao


## 3. Funcoes de leitura, segurança e desenho dos painéis

O código completo esta nesta célula. O leitor utiliza o `load_data.csv` atual e o `barcode_platemap.csv` da Cell Painting Gallery para associar placa, réplica, poço, campo e canal. A numeração dos canais não é presumida a partir dos nomes das imagens. O nome de uma pasta biologicamente ambigua não é convertido em placa por aproximacao.

A identidade EOS/controle e a dose numérica de cada poço também devem concordar com o platemap publico. Uma mudança documental não sera corrigida por suposicao.

URLs ausentes, duas imagens para o mesmo campo/canal ou associações contraditorias interrompem a etapa e ficam documentadas. Não se usa a primeira correspondencia como substituicao arbitraria.


In [ ]:
"""Read-only review of Cell Painting wells: no new fit, scoring rule or R4.
All web objects must come from cpg0036 in the public Cell Painting Gallery.
"""
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlparse, unquote, quote
import hashlib, io, json, re, time, zipfile, xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import requests
from PIL import Image, ImageDraw, ImageFont
import tifffile

PROJECT='cpg0036-EU-OS-bioactives'
BUCKET='cellpainting-gallery'
HOST='https://cellpainting-gallery.s3.amazonaws.com'
SITES=['IMTM','FMP','MEDINA','USC']
CHANNELS=['DNA','ER','AGP','Mito']
REPS=['R1','R2','R3']
CASE_CODES=['EOS100315','EOS102190','EOS100964','EOS102355','EOS100147','EOS102419']

def require(ok,msg):
    if not bool(ok): raise ValueError(msg)

def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()

def jsave(p,x):
    Path(p).write_text(json.dumps(x,ensure_ascii=False,indent=2,default=str),encoding='utf-8')

def safe_extract(path,dest):
    dest=Path(dest).resolve();dest.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(path) as z:
        for n in z.namelist():
            require((dest/n).resolve().is_relative_to(dest),'ZIP com caminho inseguro')
        z.extractall(dest)

def normwell(x):
    m=re.fullmatch(r'([A-Pa-p])0*([1-9]|1[0-9]|2[0-4])',str(x).strip())
    require(m is not None,'Poco fora do formato A01-P24: '+str(x))
    return m[1].upper()+f'{int(m[2]):02d}'

def canonical_plate(x):
    """Only explicit B1001_R1 / B1001R1 / R1B1001; not Bioactives_1_plate_1."""
    s=str(x)
    a=re.findall(r'(B\d{4})[_-]?(R[1-4])',s,re.I)
    b=[(v[1],v[0]) for v in re.findall(r'(R[1-4])[_-]?(B\d{4})',s,re.I)]
    hits=set((p.upper(),r.upper()) for p,r in a+b)
    require(len(hits)<=1,'Mais de uma placa/replica no identificador: '+s)
    return next(iter(hits)) if hits else None

def natural(x):
    return tuple((0,int(t)) if t.isdigit() else (1,t.lower()) for t in re.split(r'(\d+)',str(x)) if t)

def normfield(x):
    s=str(x).strip()
    if re.fullmatch(r'\d+(\.0+)?',s): return str(int(float(s)))
    require(bool(s),'Campo vazio no load_data')
    return s

def make_plan(sources,mode='piloto',codes=None,replicas=None):
    src=Path(sources)
    cases=pd.read_csv(src/'pocos_casos_R1_R3.csv')
    controls=pd.read_csv(src/'DMSO_R1_R3.csv')
    for d in [cases,controls]:
        require(set(d.replica)<=set(REPS),'R4 nao pertence a esta etapa')
        d['poco']=d.poco.map(normwell)
        require(not d.duplicated(['site','placa','replica','poco']).any(),'Poço repetido')
    if codes is None: codes=['EOS100315'] if mode=='piloto' else CASE_CODES
    if replicas is None: replicas=['R1'] if mode=='piloto' else REPS
    require(set(replicas)<=set(REPS),'Nao abrir R4 nesta etapa')
    d=cases[cases.composto.isin(codes)&cases.replica.isin(replicas)].copy()
    require(len(d)>0,'Nenhum caso disponivel para esta selecao')
    cs=[]
    for (s,p,r),g in d.groupby(['site','placa','replica']):
        q=controls[(controls.site==s)&(controls.placa==p)&(controls.replica==r)].copy()
        require(len(q)>=2,'Menos de dois DMSO em '+str((s,p,r)))
        q['ordem']=q.poco.map(lambda w:hashlib.sha256(f'20261003|{s}|{p}|{r}|{w}'.encode()).hexdigest())
        q=q.sort_values('ordem').head(2).drop(columns='ordem')
        q['tipo']='controle_DMSO';q['nome']='DMSO';cs.append(q)
    d=pd.concat([d]+cs,ignore_index=True)
    d['sample_id']=d.apply(lambda r:'S'+hashlib.sha256(f'{r.site}|{r.placa}|{r.replica}|{r.poco}'.encode()).hexdigest()[:12],axis=1)
    d['criterio_controles']='2 DMSO por placa/replica por ordem SHA256; sem usar intensidade, contagem ou escore'
    require(d.sample_id.is_unique,'Colisao de identificadores')
    return d.sort_values(['site','placa','replica','sample_id']).reset_index(drop=True)

class PublicGallery:
    def __init__(self,cache,timeout=45):
        self.cache=Path(cache);self.cache.mkdir(parents=True,exist_ok=True)
        self.s=requests.Session();self.timeout=timeout
    def request(self,method,url,**kwargs):
        for attempt in range(3):
            try:
                r=self.s.request(method,url,timeout=self.timeout,**kwargs)
                r.raise_for_status();return r
            except requests.RequestException:
                if attempt==2:raise
                time.sleep(2**attempt)
    def list(self,prefix,max_items=2000):
        require(prefix.startswith(PROJECT+'/'),'Prefixo nao autorizado')
        params={'list-type':'2','prefix':prefix,'max-keys':'1000'};out=[]
        while True:
            r=self.request('GET',HOST,params=params)
            root=ET.fromstring(r.content);ns={'s':'http://s3.amazonaws.com/doc/2006-03-01/'}
            for el in root.findall('s:Contents',ns):
                out.append({'key':el.findtext('s:Key',namespaces=ns),'bytes':int(el.findtext('s:Size',namespaces=ns)), 'etag':el.findtext('s:ETag',namespaces=ns)})
            require(len(out)<=max_items,'Listagem excedeu limite de seguranca; nenhuma imagem baixada')
            if root.findtext('s:IsTruncated',namespaces=ns)!='true':break
            token=root.findtext('s:NextContinuationToken',namespaces=ns)
            require(token,'Paginacao sem token');params['continuation-token']=token
        return out
    def bytes(self,key,max_bytes=32*1024*1024):
        p=self.cache/(hashlib.sha256(key.encode()).hexdigest()+'.bin')
        if p.exists():return p.read_bytes()
        info=self.head(key);require(info['bytes']<=max_bytes,'Objeto de metadados muito grande: '+key)
        r=self.request('GET',object_url(key),stream=True);buf=io.BytesIO();n=0
        for part in r.iter_content(1024*1024):
            n+=len(part);require(n<=max_bytes,'Transferencia excedeu limite');buf.write(part)
        blob=buf.getvalue();require(len(blob)==info['bytes'],'Transferencia incompleta');p.write_bytes(blob)
        return blob
    def head(self,key):
        r=self.request('HEAD',object_url(key));n=r.headers.get('Content-Length')
        require(n is not None,'HEAD sem tamanho: '+key)
        return {'bytes':int(n),'etag':r.headers.get('ETag',''),'last_modified':r.headers.get('Last-Modified','')}

def object_url(key):
    require(key.startswith(PROJECT+'/') and '..' not in Path(key).parts,'Chave nao permitida')
    return HOST+'/'+quote(key,safe='/')

def object_key(text,site):
    text=str(text).strip().replace('\\','/')
    require(bool(text) and text.lower()!='nan','URL vazia')
    if text.startswith('s3://'):
        u=urlparse(text);require(u.netloc==BUCKET,'Bucket diferente');key=unquote(u.path.lstrip('/'))
    elif text.startswith(('http://','https://')):
        u=urlparse(text);require(u.scheme=='https' and (u.netloc==BUCKET+'.s3.amazonaws.com' or u.netloc==BUCKET+'.s3.us-east-1.amazonaws.com'),'Host nao autorizado')
        key=unquote(u.path.lstrip('/'));require(not u.query,'URL com query inesperada')
    elif PROJECT+'/' in text:
        key=text[text.index(PROJECT+'/'):]
    else:raise ValueError('Caminho sem prefixo documental reconhecido: '+text)
    require(key.startswith(f'{PROJECT}/{site}/images/') and '..' not in Path(key).parts,'Imagem fora do local autorizado: '+key)
    return key

def read_remote_table(blob):
    # No executable content: read textual CSV/TSV only.
    return pd.read_csv(io.BytesIO(blob),sep=None,engine='python',dtype=str,keep_default_na=False)

def barcode_lookup(gallery,site):
    out={};records=[]
    keys=gallery.list(f'{PROJECT}/{site}/workspace/metadata/',max_items=1500)
    for ob in keys:
        k=ob['key']
        if 'hepg2' not in k.lower() or not k.lower().endswith('barcode_platemap.csv'):continue
        blob=gallery.bytes(k);d=read_remote_table(blob)
        require({'Assay_Plate_Barcode','Plate_Map_Name'}<=set(d),'barcode_platemap sem colunas esperadas')
        for _,r in d.iterrows():
            c=canonical_plate(r.Plate_Map_Name)
            if c is None:continue
            batch=k.split('/platemaps/')[-1].split('/')[0]
            key=(batch,r.Assay_Plate_Barcode)
            require(key not in out or out[key]==c,'Barcode ambiguo')
            out[key]=c
            map_candidates=[v['key'] for v in keys if v['key'].startswith(str(Path(k).parent)+'/platemap/') and Path(v['key']).stem==str(r.Plate_Map_Name) and Path(v['key']).suffix.lower() in ['.txt','.tsv','.csv']]
            records.append({'site':site,'barcode':r.Assay_Plate_Barcode,'batch':batch,'placa':c[0],'replica':c[1],'fonte':k,'sha256':hashlib.sha256(blob).hexdigest(),'platemap_key':map_candidates[0] if len(map_candidates)==1 else ''})
    return out,records

def validate_public_platemap(df,plan,site,identity,key):
    """Confirm well -> EOS/control and dose against the public map; no name matching."""
    def col(allowed):
        found=[v for v in allowed if v in df]
        require(len(found)==1,'Coluna do platemap ausente/ambigua: '+str(allowed))
        return found[0]
    wc=col(['well_position','Metadata_Well']);ec=col(['Metadata_EOS','compound_id'])
    dc=col(['Metadata_Concentration','Concentration'])
    d=df.copy();d['poco']=d[wc].map(normwell)
    require(not d.poco.duplicated().any(),'Poco duplicado no platemap publico')
    p,r=identity;g=plan[(plan.site==site)&(plan.placa==p)&(plan.replica==r)]
    mapped=d.set_index('poco');rows=[]
    for _,row in g.iterrows():
        require(row.poco in mapped.index,'Poco ausente do platemap publico: '+row.poco)
        v=mapped.loc[row.poco]
        require(str(v[ec]).strip()==str(row.composto),'Identidade do composto diverge do mapa publico: '+str((site,p,r,row.poco)))
        a,b=float(v[dc]),float(row.dose)
        require(np.isfinite(a) and a==b,'Concentracao numerica diverge do mapa publico; nao corrigir automaticamente')
        rows.append({'site':site,'placa':p,'replica':r,'poco':row.poco,'composto':row.composto,'dose_numerica':b,'platemap_key':key,'resultado':'identidade_e_dose_documentais_concordantes','nota':'Unidade numerica preservada; nao inferida a partir do nome do arquivo'})
    return rows

def infer_identity(key,aliases):
    explicit=canonical_plate(key)
    pieces=key.split('/');candidates=set()
    for (batch,barcode),v in aliases.items():
        if batch in pieces and barcode in pieces:candidates.add(v)
    if candidates:
        require(len(candidates)==1,'Mais de um barcode corresponde ao load_data')
        return next(iter(candidates)) # authority is barcode table; not microscope basename
    return explicit

def parse_load_data(df,site,identity,key,aliases):
    p,r=identity
    if r not in REPS:return pd.DataFrame()
    require({'Metadata_Well','Metadata_Site'}<=set(df),'load_data sem Well/Site: '+key)
    rows=[];batch=key.split('/load_data_csv/')[-1].split('/')[0]
    for i,row in df.iterrows():
        if 'Metadata_Plate' in df and str(row.Metadata_Plate):
            raw=str(row.Metadata_Plate).strip();resolved=aliases.get((batch,raw)) or canonical_plate(raw)
            if resolved is None and raw.upper()==p:resolved=identity  # replica is documented in this load_data directory
            require(resolved is not None and resolved==identity,'Placa do load_data nao confere: '+key+' / '+raw)
        w=normwell(row.Metadata_Well);field=normfield(row.Metadata_Site)
        for ch in CHANNELS:
            cols=[c for c in ['URL_Orig'+ch,'URL_'+ch] if c in df and str(row[c]).strip()]
            if cols:
                ks={object_key(row[c],site) for c in cols};require(len(ks)==1,'URLs de canal conflitantes')
                k=next(iter(ks));source=cols[0]
            else:
                pairs=[('PathName_Orig'+ch,'FileName_Orig'+ch),('PathName_'+ch,'FileName_'+ch)]
                pairs=[v for v in pairs if v[0] in df and v[1] in df and row[v[1]]]
                require(len(pairs)==1,'Canal nao documentado sem ambiguidade: '+ch)
                pc,fc=pairs[0];k=object_key(row[pc].rstrip('/')+'/'+row[fc],site);source=pc+' + '+fc
            rows.append({'site':site,'placa':p,'replica':r,'poco':w,'campo':field,'canal':ch,'key':k,'fonte_load_data':key,'linha_load_data':i+2,'coluna_canal':source})
    result=pd.DataFrame(rows)
    cols=['site','placa','replica','poco','campo','canal']
    require(not result.duplicated(cols).any(),'Mais de uma imagem para o mesmo campo/canal: '+key)
    return result

def discover_images(plan,gallery,out):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    allrows=[];audit=[];errors=[];barcodes=[];mapaudit=[]
    for site,g in plan.groupby('site'):
        wanted=set(zip(g.placa,g.replica));aliases,br=barcode_lookup(gallery,site);barcodes+=br
        objs=gallery.list(f'{PROJECT}/{site}/workspace/load_data_csv/',max_items=1500)
        keys=[x['key'] for x in objs if 'hepg2' in x['key'].lower() and x['key'].lower().endswith('.csv')]
        # Prefer current load_data.csv; do not touch load_data_csv_orig or use guessed channel numbers.
        keys=[k for k in keys if k.endswith('/load_data.csv') or k.endswith('/load_data_with_illum.csv')]
        matched={}
        for k in keys:
            ident=infer_identity(k,aliases)
            if ident not in wanted:continue
            matched.setdefault(ident,[]).append(k)
        for ident in sorted(wanted):
            try:
                candidates=matched.get(ident,[])
                base=[k for k in candidates if k.endswith('/load_data.csv')]
                if base:candidates=base
                require(len(candidates)==1,f'{site} {ident}: esperado um load_data, encontrados {len(candidates)}. Rever barcode, nao adivinhar.')
                mapkeys={v['platemap_key'] for v in br if (v['placa'],v['replica'])==ident and v.get('platemap_key')}
                require(len(mapkeys)==1,f'{site} {ident}: platemap publico unico nao resolvido; nao presumir EOS pelo nome da placa')
                mk=next(iter(mapkeys));mb=gallery.bytes(mk)
                checks=validate_public_platemap(read_remote_table(mb),plan,site,ident,mk)
                for chk in checks:chk['platemap_sha256']=hashlib.sha256(mb).hexdigest()
                mapaudit+=checks
                key=candidates[0];blob=gallery.bytes(key);df=read_remote_table(blob)
                table=parse_load_data(df,site,ident,key,aliases)
                table=table[table.poco.isin(g[(g.placa==ident[0])&(g.replica==ident[1])].poco)]
                allrows.append(table);audit.append({'site':site,'placa':ident[0],'replica':ident[1],'key':key,'sha256':hashlib.sha256(blob).hexdigest(),'n_linhas':len(df)})
            except Exception as e:errors.append({'site':site,'placa':ident[0],'replica':ident[1],'erro':str(e)})
    pd.DataFrame(audit).to_csv(out/'load_data_fontes.csv',index=False)
    pd.DataFrame(barcodes).to_csv(out/'barcodes_fontes.csv',index=False)
    pd.DataFrame(mapaudit).to_csv(out/'platemap_publico_conferencia.csv',index=False)
    pd.DataFrame(errors,columns=['site','placa','replica','erro']).to_csv(out/'pendencias_metadados.csv',index=False)
    require(not errors,'Correspondencia com imagens pendente; consulte pendencias_metadados.csv. Nenhum campo foi substituido.')
    require(allrows,'Nenhum load_data compativel encontrado')
    data=pd.concat(allrows,ignore_index=True)
    require(not data.duplicated(['site','placa','replica','poco','campo','canal']).any(),'Imagens duplicadas entre arquivos')
    return data

def choose_fields(data,plan,n_fields=1):
    require(n_fields>=1,'Numero de campos invalido');parts=[];audit=[]
    for (s,p,r),g in plan.groupby(['site','placa','replica']):
        t=data[(data.site==s)&(data.placa==p)&(data.replica==r)]
        sets=[]
        for w in g.poco:
            q=t[t.poco==w];ok=q.groupby('campo').canal.agg(set)
            fields=set(ok[ok.map(lambda a:a==set(CHANNELS))].index)
            require(fields,f'Nenhum campo completo de 4 canais: {s}/{p}/{r}/{w}')
            sets.append(fields)
        common=sorted(set.intersection(*sets),key=natural)
        require(len(common)>=n_fields,f'{s}/{p}/{r}: poucos campos comuns. Nao substituir campos silenciosamente.')
        inds=np.unique(np.round(np.linspace(0,len(common)-1,n_fields)).astype(int));chosen=[common[i] for i in inds]
        if n_fields==1:chosen=[common[0]]
        parts.append(t[t.campo.isin(chosen)])
        audit.append({'site':s,'placa':p,'replica':r,'campos_disponiveis_em_comum':','.join(common),'campos_escolhidos':','.join(chosen),'criterio':'primeiro, ou indices igualmente espacados na lista natural; independente da imagem'})
    selected=pd.concat(parts,ignore_index=True).merge(plan,on=['site','placa','replica','poco'],validate='many_to_one')
    return selected,pd.DataFrame(audit)

def prepare_download(data,gallery,out,max_total_mb=1024,max_file_mb=128):
    rows=[];cache={}
    for k in data.key.unique():
        info=gallery.head(k);require(info['bytes']<=max_file_mb*1024**2,'Imagem excede limite individual')
        cache[k]=info
    for _,r in data.iterrows(): rows.append({**r.to_dict(),**cache[r.key]})
    result=pd.DataFrame(rows)
    total=sum(v['bytes'] for v in cache.values())
    result.to_csv(Path(out)/'manifesto_imagens.csv',index=False)
    jsave(Path(out)/'tamanho_previsto.json',{'n_objetos_unicos':len(cache),'bytes':total,'limite_MB':max_total_mb})
    require(total<=max_total_mb*1024**2,f'Plano exige {total/1024**2:.1f} MiB; limite {max_total_mb}. Rever volume antes de baixar.')
    return result

def download_images(manifest,gallery,out):
    out=Path(out);dest=out/'imagens_originais';dest.mkdir(parents=True,exist_ok=True);rows=[]
    for k,g in manifest.groupby('key',sort=True):
        info=g.iloc[0];suffix=Path(k).suffix.lower();suffix=suffix if suffix in ['.tif','.tiff','.png'] else '.dat'
        path=dest/(hashlib.sha256(k.encode()).hexdigest()+suffix);record=path.with_suffix(path.suffix+'.json')
        if path.exists():
            require(record.exists(),'Arquivo sem registro de integridade: '+str(path))
            saved=json.loads(record.read_text());require(digest(path)==saved['sha256'],'Imagem local modificada')
            require(saved['etag']==info.etag and saved['bytes']==int(info.bytes),'Objeto remoto mudou; usar nova execucao')
        else:
            tmp=path.with_suffix('.part');n=0
            try:
                res=gallery.request('GET',object_url(k),stream=True)
                require(res.headers.get('ETag','')==info.etag,'ETag mudou desde o planejamento')
                with tmp.open('wb') as f:
                    for b in res.iter_content(1024*1024):
                        n+=len(b);require(n<=int(info.bytes),'Imagem excedeu tamanho anunciado');f.write(b)
                require(n==int(info.bytes),'Imagem incompleta');tmp.replace(path)
                saved={'key':k,'sha256':digest(path),'etag':info.etag,'bytes':n,'utc':datetime.now(timezone.utc).isoformat()};jsave(record,saved)
            finally:
                if tmp.exists():tmp.unlink()
        rows.append({'key':k,'arquivo_local':str(path),'sha256':saved['sha256']})
    return manifest.merge(pd.DataFrame(rows),on='key',validate='many_to_one')

def read_plane(path):
    if Path(path).suffix.lower()=='.png': a=np.asarray(Image.open(path))
    else:a=tifffile.imread(path)
    a=np.squeeze(a)
    require(a.ndim==2,'Imagem nao e um plano 2D. Nao fazer projecao/corte automatico: '+str(path))
    require(np.isfinite(a).all(),'Pixels nao finitos')
    return a

def reference_limits(arrays,q=(0.5,99.5)):
    # Equal deterministic sampling per control field; display only, no preprocessing of profiles.
    v=[]
    for a in arrays:
        f=np.asarray(a).ravel();n=min(len(f),100000)
        v.append(f[np.linspace(0,len(f)-1,n,dtype=int)])
    require(v,'Sem DMSO para escala de exibicao')
    lo,hi=np.percentile(np.concatenate(v),q)
    require(np.isfinite(lo+hi) and hi>lo,'DMSO com faixa nula; nao ajustar pelos tratamentos')
    return float(lo),float(hi)

def display_plane(a,lo,hi):
    require(hi>lo,'Limites de contraste invalidos')
    b=(np.clip((a.astype(float)-lo)/(hi-lo),0,1)*255).round().astype('uint8')
    return Image.fromarray(b),float(np.mean(a<lo)),float(np.mean(a>hi))

def render_review(manifest,out):
    out=Path(out);panels=out/'paineis_mascarados';panels.mkdir(exist_ok=True)
    scales=[];diagnostics=[];records=[]
    font=ImageFont.load_default(size=16)
    for (s,p,r),g in manifest.groupby(['site','placa','replica']):
        limits={}
        for ch in CHANNELS:
            ctrl=g[(g.tipo=='controle_DMSO')&(g.canal==ch)]
            limits[ch]=reference_limits([read_plane(v) for v in ctrl.arquivo_local])
            scales.append({'site':s,'placa':p,'replica':r,'canal':ch,'min_exibicao':limits[ch][0],'max_exibicao':limits[ch][1],'origem':'percentis 0.5/99.5 dos DMSO selecionados; mesma placa/replica/canal'})
        for field,h in g.groupby('campo',sort=True):
            ids=sorted(h.sample_id.unique())
            for crop in [False,True]:
                cw,chh=290,320;img=Image.new('RGB',(cw*len(CHANNELS),70+chh*len(ids)),(255,255,255));draw=ImageDraw.Draw(img)
                title=f'{s} {p} {r} | campo {field} | '+('recorte central 512 px' if crop else 'campo inteiro reduzido')
                draw.text((8,4),title,fill=(0,0,0),font=font)
                draw.text((8,28),'Escala DMSO comum por canal; sem escala fisica entre locais. IDs mascarados.',fill=(0,0,0),font=font)
                for i,sid in enumerate(ids):
                    q=h[h.sample_id==sid]
                    for j,channel in enumerate(CHANNELS):
                        rr=q[q.canal==channel];require(len(rr)==1,'Campo/canal incompleto')
                        row=rr.iloc[0];a=read_plane(row.arquivo_local);height,width=a.shape
                        lo,hi=limits[channel];b,below,above=display_plane(a,lo,hi)
                        if crop:
                            size=min(512,height,width);left=(width-size)//2;top=(height-size)//2
                            b=b.crop((left,top,left+size,top+size))
                        else:left=top=0;size=0
                        b.thumbnail((280,270),resample=Image.Resampling.LANCZOS)
                        x=j*cw;y=70+i*chh;img.paste(b,(x+5,y+40))
                        draw.text((x+5,y),sid+' | '+channel,fill=(0,0,0),font=font)
                        draw.text((x+5,y+20),f'{width}x{height} px; clip alto {100*above:.1f}%',fill=(0,0,0),font=font)
                        if not crop:
                            dtype=a.dtype;imax=np.iinfo(dtype).max if dtype.kind in 'iu' else None
                            diagnostics.append({'sample_id':sid,'site':s,'placa':p,'replica':r,'campo':field,'canal':channel,'altura_px':height,'largura_px':width,'dtype':str(dtype),'fracao_clip_exibicao_alta':above,'fracao_clip_exibicao_baixa':below,'fracao_max_dtype':float(np.mean(a==imax)) if imax is not None else np.nan,'nota':'max_dtype nao e saturacao real do detector sem metadados; contagem de objetos refere-se ao poco, nao ao campo'})
                    if not crop:
                        records.append({'sample_id':sid,'site':s,'placa':p,'replica':r,'campo':field,'foco':'','fundo_nao_uniforme':'','marcacao_pontual_ou_tubular':'','distribuicao_perinuclear':'','densidade_visual':'','detritos':'','avaliavel_visualmente':'','observacoes':'','revisor':'','data_revisao':''})
                name=f'{s}_{p}_{r}_campo{field}_'+('crop' if crop else 'inteiro')+'.png';img.save(panels/name)
    pd.DataFrame(scales).to_csv(out/'escalas_exibicao_DMSO.csv',index=False)
    pd.DataFrame(diagnostics).to_csv(out/'diagnosticos_pixels.csv',index=False)
    # Never overwrite manual review, even when render is resumed.
    target=out/'avaliacao_visual_PREENCHER.csv'
    if not target.exists():pd.DataFrame(records).to_csv(target,index=False)
    pd.DataFrame(records).to_csv(out/'avaliacao_visual_modelo.csv',index=False)
    manifest[['sample_id','site','placa','replica','poco','composto','nome','tipo']].drop_duplicates().to_csv(out/'chave_identidades_ABRIR_DEPOIS.csv',index=False)
    return sorted(panels.glob('*.png'))


In [ ]:
require(FONTES_ZIP.is_file(),'Pacote de fontes ausente: '+str(FONTES_ZIP))
require(digest(FONTES_ZIP)=='4128c1ee872ba5080264ad9d1db2cacaf34425f93220d99d8a46ccc63bda46a1','Pacote de fontes diferente da entrega; nao misturar versoes')
CFG={'codigo_sha256':'41bb2969ee0d5d1d3ef132718080c319ca6e33ba4b6460ecd451d9e9e0b4536f','modo':MODO,'extremos_QC':INCLUIR_EXTREMOS_QC,'limite_MB':LIMITE_MB,'fontes_sha256':digest(FONTES_ZIP),'R4':False}
SAIDA=Path(RETOMAR_PASTA) if RETOMAR_PASTA else RAIZ/'resultados'/('inspecao_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
SAIDA.mkdir(parents=True,exist_ok=True)
if (SAIDA/'protocolo.json').exists():require(json.loads((SAIDA/'protocolo.json').read_text())==CFG,'Protocolo alterado: inicie outra execucao')
else:jsave(SAIDA/'protocolo.json',CFG)
FONTES=SAIDA/'fontes';safe_extract(FONTES_ZIP,FONTES)
for name,h in json.loads((FONTES/'manifesto_arquivos.json').read_text()).items():require(digest(FONTES/name)==h,'Fonte alterada: '+name)
jsave(SAIDA/'versoes.json',VERSOES)
ESTADO={'dados_locais':'conferidos','imagens':'nao_baixadas','leitura_humana':'pendente','R4_lida':False}
print('SAIDA:',SAIDA)


## 4. Quais poços serao vistos?

Antes de abrir as imagens, congelamos suas coordenadas. O piloto contém 12 poços: quatro com Alda-1 e oito DMSO. Os IDs dos painéis não mostram composto ou status; a chave separada permite uma leitura inicialmente mascarada. Isso não garante cegamento completo, pois o protocolo e os arquivos continuam acessiveis.

A tabela de contagem refere-se ao poço agregado do processamento anterior, e não ao número de objetos do campo que sera exibido. Poucos objetos ou uma medida dominante são alertas; não demonstram toxicidade ou erro de segmentacao.


In [ ]:
codes=None
if MODO=='completo' and INCLUIR_EXTREMOS_QC:
    codes=sorted(set(CASE_CODES)|set(pd.read_csv(FONTES/'casos_qualidade_selecao.csv').composto))
PLANO=make_plan(FONTES,MODO,codes=codes)
PLANO.to_csv(SAIDA/'plano_pocos.csv',index=False)
for site,g in PLANO.groupby('site'):
    mapa=pd.read_csv(FONTES/f'mapa_{site}_R1_R3.csv').rename(columns={'Metadata_Plate':'placa','Metadata_Batch':'replica','Metadata_Well':'poco','Metadata_EOS':'composto','Metadata_Concentration':'dose'})
    mapa['poco']=mapa.poco.map(normwell)
    chk=g.merge(mapa[['placa','replica','poco','composto','dose']],on=['placa','replica','poco'],suffixes=('_plano','_mapa'),validate='one_to_one')
    require(len(chk)==len(g) and chk.composto_plano.eq(chk.composto_mapa).all(),'Composto nao confere no mapa')
    require(np.allclose(chk.dose_plano,chk.dose_mapa),'Dose nao confere no mapa')
display(PLANO.groupby(['site','tipo']).size().rename('n_pocos').reset_index())
print('Plano completo salvo em plano_pocos.csv; sem consultar as imagens.')


## 5. Localizar os arquivos originais na Cell Painting Gallery

A data de processamento no CSV agregado não sera usada para adivinhar a pasta do microscopio. Buscamos a ligacao nos metadados publicos atuais do projeto `cpg0036-EU-OS-bioactives`. Não ha necessidade de conta AWS, credenciais ou download de uma placa inteira.

Existem relatos documentados de correção de caminhos e de identificação no IMTM. Por isso, usamos o `load_data_csv` atual, não `load_data_csv_orig`, mantemos os hashes e não aplicamos uma segunda troca de placas. Esta etapa não inclui R4. A presença de um caminho não garante a existencia da imagem; a célula seguinte verifica cada objeto.

Se esta etapa parar, compartilhe `falha_acesso.json` e `pendencias_metadados.csv`, quando gerado. Não altere números de canal ou nomes de arquivo por tentativa e erro.


In [ ]:
GALERIA=PublicGallery(SAIDA/'cache_metadados')
try:
    IMAGENS=discover_images(PLANO,GALERIA,SAIDA)
    IMAGENS.to_csv(SAIDA/'imagens_disponiveis_pocos.csv',index=False)
    ESTADO['metadados_remotos']='resolvidos'
    print('Correspondencias documentais:',len(IMAGENS))
except Exception as e:
    ESTADO['imagens']='pendente_acesso_ou_mapeamento'
    jsave(SAIDA/'falha_acesso.json',{'erro':str(e),'estado':ESTADO})
    raise RuntimeError('Acesso ou identidade das imagens pendente. Os resultados cientificos anteriores nao foram modificados. Detalhe: '+str(e)) from e


## 6. Congelar campos e conferir o volume antes do download

Um campo precisa conter os quatro canais e estar disponivel nos casos e controles correspondentes. No piloto escolhemos o menor ID de campo na ordenacao natural; no lote completo, três IDs igualmente espaçados. Um campo ausente não é substituído por outro visualmente favoravel.

A seleção por ID é uma regra reprodutivel de amostragem; o mesmo ID não significa a mesma posicao fisica em diferentes microscopios. O tamanho total é obtido dos objetos remotos e os seus ETags são registrados.


In [ ]:
N_CAMPOS=1 if MODO=='piloto' else 3
ESCOLHIDAS,AUD_CAMPOS=choose_fields(IMAGENS,PLANO,N_CAMPOS)
AUD_CAMPOS.to_csv(SAIDA/'campos_escolhidos.csv',index=False)
MANIFESTO=prepare_download(ESCOLHIDAS,GALERIA,SAIDA,max_total_mb=LIMITE_MB)
display(AUD_CAMPOS)
print(json.loads((SAIDA/'tamanho_previsto.json').read_text()))


## 7. Download delimitado e verificacao de integridade

Somente os arquivos do manifesto são baixados. A retomada reutiliza uma imagem apenas se tamanho, ETag e hash forem compativeis. Os arquivos originais são preservados e nunca recebem ajustes de contraste. Não existe solicitacao recursiva de toda a colecao de imagens.


In [ ]:
require(BAIXAR_IMAGENS,'Download desativado: mude BAIXAR_IMAGENS para True somente apos conferir o plano')
BAIXADAS=download_images(MANIFESTO,GALERIA,SAIDA)
BAIXADAS.to_csv(SAIDA/'imagens_baixadas.csv',index=False)
ESTADO['imagens']='baixadas_integridade_conferida'
print('Imagens unicas baixadas:',BAIXADAS.key.nunique())


## 8. Painéis para leitura visual

Cada canal é exibido em escala de cinza. Os limites são os percentis 0,5 e 99,5 dos pixels amostrados dos dois DMSO da mesma placa/réplica/canal. Esses limites são aplicados igualmente a todos os tratamentos da respectiva placa. A fracao de pixels fora da faixa de exibição fica registrada; clipping no painel não equivale a saturação do detector.

Sao gerados um painel do campo inteiro e outro com o recorte central fixo de até 512 pixels. Não escolhemos a célula mais interessante. Uma regiao central vazia permanece vazia. O arquivo TIFF conserva a resolução original.

As imagens são **brutas**, sem reproduzir a correção de iluminação ou a segmentacao do processamento que produziu os perfis. Os painéis não exibem máscaras de segmentacao. Comparacoes de brilho ou tamanho absoluto entre locais não são validas sem calibracao de exposicao e tamanho do pixel. Não foi desenhada uma barra de micrômetros com tamanho presumido.


In [ ]:
PAINEIS=render_review(BAIXADAS,SAIDA)
ESTADO['paineis']='gerados';jsave(SAIDA/'estado.json',ESTADO)
for p in PAINEIS:
    display(Markdown('### '+p.stem))
    display(IPyImage(filename=str(p)))
print('Anote em avaliacao_visual_PREENCHER.csv. A rotina nao preenche resultados biologicos por voce.')


## 9. Registrar a observação antes de consultar o nome

Preencha foco, fundo, densidade visual, detritos, organização pontual ou tubular da marcação e sua distribuição. Use 'não avaliavel' quando a resolução ou o fundo não permitir decidir, em vez de forcar uma classificacao. Nenhum desses campos sera convertido automaticamente em filtro da análise anterior.

Anote quem avaliou e em que data. Conserve a primeira leitura antes de abrir `chave_identidades_ABRIR_DEPOIS.csv`. A ficha e criada vazia e nunca e sobrescrita pela geracao dos painéis. Os campos não constituem uma escala biológica validada e as observacoes dos campos de um mesmo poço não são réplicas experimentais independentes.


## 10. Voltar aos valores com sinal

O RMS soma desvios ao quadrado e perde o sinal. Para investigar uma alteração especifica, consulte as mesmas medidas, nas três réplicas e nos quatro locais. A tabela incluida contém os valores MAD locais ja calculados, sem nova normalizacao. Para os seis casos, todas as medidas selecionadas em cada local foram preservadas; 114 delas são comuns.

O exemplo abaixo foi escolhido por ser uma medida de distribuição radial presente nos quatro conjuntos e destacada na interpretação anterior. Um valor positivo significa desvio acima da mediana DMSO local dessa caracteristica; não aumento de função mitocondrial. A mesma magnitude global pode coexistir com sinais diferentes. Não construimos aqui um escore comum nem um novo ranking de candidatos.


In [ ]:
MEDIDAS=pd.read_csv(FONTES/'medidas_casos_com_sinal.csv')
MEDIDA_EXEMPLO='Cells_RadialDistribution_MeanFrac_Mito_3of4'
EXEMPLO=MEDIDAS[(MEDIDAS.composto=='EOS100315')&(MEDIDAS.medida==MEDIDA_EXEMPLO)]
display(EXEMPLO[['site','nome','medida','z_R1','z_R2','z_R3','mediana_z','sinal_3_concordante']])
EXEMPLO.to_csv(SAIDA/'exemplo_Alda1_com_sinal.csv',index=False)
# Para outro caso, altere somente a selecao da tabela, nao os valores ou o criterio original.


## 11. Fechamento: o que a etapa permite concluir

A identidade dos arquivos, a presença dos canais e a qualidade visual devem ser avaliadas antes da descricao de um fenotipo. Textura ou distribuição radial não demonstram, isoladamente, fissão, fusão, apoptose, toxicidade ou alteração bioenergetica. A anotacao de um alvo farmacologico também não prova que ele foi modulado nas condições analisadas.

O fechamento desta etapa deve distinguir: identidade resolvida; imagens baixadas; leitura visual realizada; e hipótese biológica. O relatório abaixo não declara validação biológica automatica. O pacote de leitura inclui painéis, manifestos e fichas, mas exclui os TIFFs originais para evitar um ZIP excessivo; seus hashes e caminhos permanecem registrados.

No primeiro retorno, envie o relatório desta célula e os painéis do piloto. A partir da conferência, podemos ampliar para R1-R3 e os casos discordantes, sem escolher campos pela aparencia.


In [ ]:
import shutil
review=SAIDA/'avaliacao_visual_PREENCHER.csv'
n_preenchidos=0
if review.exists():
    r=pd.read_csv(review).fillna('')
    if 'avaliavel_visualmente' in r:n_preenchidos=int(r.avaliavel_visualmente.astype(str).str.strip().ne('').sum())
ESTADO['n_fichas_com_avaliabilidade_preenchida']=n_preenchidos
ESTADO['validacao_biologica']='nao realizada'
report=f"""# Inspecao de imagens e perfis: {MODO}
Pocos planejados: {len(PLANO)}. Imagens baixadas: {BAIXADAS.key.nunique()}.
Paineis: {len(PAINEIS)}. Fichas com avaliabilidade preenchida: {n_preenchidos}.
R4 nao lida. Nenhum modelo treinado, escore redefinido ou candidato reclassificado.
Imagens brutas; contraste de exibicao comum estimado nos DMSO por placa/replica/canal.
Nenhuma correcao de iluminacao, segmentacao ou interpretacao funcional foi aplicada.
A leitura humana e as conclusoes biologicas ainda exigem conferencia.
"""
(SAIDA/'relatorio_inspecao.md').write_text(report,encoding='utf-8');jsave(SAIDA/'estado.json',ESTADO)
pack=SAIDA.with_name(SAIDA.name+'_leitura.zip')
with zipfile.ZipFile(pack,'w',zipfile.ZIP_DEFLATED) as zz:
    for p in SAIDA.rglob('*'):
        if p.is_file() and not {'imagens_originais','cache_metadados'}.intersection(p.relative_to(SAIDA).parts):zz.write(p,p.relative_to(SAIDA))
display(Markdown(report))
print('RESULTADOS:',SAIDA,'\nPACOTE:',pack)


## Fontes e proveniencia

A base privada é a comparação padronizada de quatro campanhas identificadas em `proveniencia.json`. Os valores não foram substituidos por fontes externas. Os arquivos publicos de imagem serao associados pelas tabelas documentais do projeto, com os objetos e datas efetivamente consultados registrados no manifesto.

- Repositorio dos autores: https://github.com/schmiedc/EU-OS_bioactives
- Estrutura e significado de load_data: https://broadinstitute.github.io/cellpainting-gallery/data_structure.html
- Acesso publico: https://broadinstitute.github.io/cellpainting-gallery/download_instructions.html
- Discussao tecnica das correcoes de metadados: https://github.com/broadinstitute/cellpainting-gallery/discussions/57

Essas fontes descrevem acesso e organização; não substituem a verificacao de cada objeto desta execução. Não foi utilizada geracao de imagens por inteligencia artificial.
